# as-strided-windowing — ex3: strided 1-D windows with step greater than one

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `as-strided-windowing`. Running the final beacon cell reports progress against the `PyTorch: as_strided windowing` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: as_strided windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`as-strided-windowing`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-windowing"
DD_SUBTOPIC = "PyTorch: as_strided windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `as_strided` — quick refresher

`t.as_strided(x, size, stride)` re-interprets `x`'s underlying storage with a new shape + stride tuple. **It never allocates.** The arguments are in *elements*, not bytes. The view aliases the same storage — writes through one alias are visible through the other.

**Window-stride vs in-window-stride.** A 1-D sliding window of width `K` stepping by 1 element has shape `(L_out, K)` and stride `(s, s)` where `s = x.stride(0)`. To dilate or sub-sample the windows, you multiply ONE of those by an integer factor:
- `stride = (s, s*d)`     → dilated window (gaps INSIDE the window)
- `stride = (s*step, s)`  → strided windows (gaps BETWEEN windows)
Compute `L_out` from the same formula PyTorch's conv uses: `L_out = (L - dilated_K) // step + 1`.

### Exercise 3 — strided 1-D windows with step greater than one

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `t.as_strided` with a multiplied outer stride to build a 1-D sliding-window view whose windows step by `step` elements between consecutive positions (the missing-`stride>1` extension of ARENA's `conv1d_minimal`).
> Keywords: sliding-window, stride-step, as_strided, subsample
> ```

**KCs targeted:** `as-strided-window-stride`, `as-strided-window-size`

Implement `ex3_strided_windows(x, K, step)`. Given a 1-D tensor `x` of length `L`, a window width `K`, and a positive integer `step`, return a zero-copy view of shape `(L_out, K)` where:

- Row `i` is `x[i*step : i*step + K]`.
- `L_out = (L - K) // step + 1`.

Use `t.as_strided` with the right `size` and `stride`. Pull the element-stride of `x` from `x.stride()` — do **not** hard-code `1` (this matters if `x` was built via `x = source[::2]`).

Required: `K >= 1`, `step >= 1`, `L >= K`. Return must alias `x`'s storage (no copy — writes propagate).

In [ ]:
def ex3_strided_windows(x: Tensor, K: int, step: int) -> Tensor:
    """Sliding windows of width K stepping by step elements."""
    raise NotImplementedError()


def _test_ex3():
    # Basic shape + value check.
    x = t.arange(10)
    w = ex3_strided_windows(x, K=3, step=2)
    assert tuple(w.shape) == (4, 3), f'expected (4,3), got {tuple(w.shape)}'
    expected = t.tensor([
        [0, 1, 2],
        [2, 3, 4],
        [4, 5, 6],
        [6, 7, 8],
    ])
    assert t.equal(w, expected), f'value mismatch:\n{w}\nvs\n{expected}'

    # Aliasing check — writes through w must propagate to x.
    x2 = t.arange(8).clone()
    w2 = ex3_strided_windows(x2, K=3, step=2)
    w2[0, 0] = -99
    assert x2[0].item() == -99, 'returned view must alias x storage'

    # step==1 collapses to the dense windowing of ex1.
    x3 = t.arange(6)
    w_dense = ex3_strided_windows(x3, K=2, step=1)
    assert tuple(w_dense.shape) == (5, 2)
    assert t.equal(w_dense, t.tensor([[0,1],[1,2],[2,3],[3,4],[4,5]]))

    # step==K → non-overlapping chunks.
    x4 = t.arange(12)
    chunks = ex3_strided_windows(x4, K=3, step=3)
    assert tuple(chunks.shape) == (4, 3)
    assert t.equal(chunks, t.tensor([[0,1,2],[3,4,5],[6,7,8],[9,10,11]]))

    # Non-unit-stride source — must use x.stride(0), not hard-coded 1.
    src = t.arange(20)
    sub = src[::2]                  # stride(0) == 2 elements, length 10
    w_sub = ex3_strided_windows(sub, K=3, step=2)
    assert tuple(w_sub.shape) == (4, 3)
    # Expected: sub == [0,2,4,6,8,10,12,14,16,18]; step-2 windows of width 3.
    exp_sub = t.tensor([[0,2,4],[4,6,8],[8,10,12],[12,14,16]])
    assert t.equal(w_sub, exp_sub), (
        f'non-unit-stride source mishandled — did you hard-code stride=1?\n'
        f'got: {w_sub}\nexpected: {exp_sub}'
    )
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_strided_windows(x: Tensor, K: int, step: int) -> Tensor:
    L = x.shape[0]
    s, = x.stride()
    L_out = (L - K) // step + 1
    return t.as_strided(x, size=(L_out, K), stride=(s * step, s))
```

**Outer stride scales by `step`, inner stride stays at `s`.** The outer stride says 'how to jump to the NEXT window'; the inner stride says 'how to walk inside one window'. The `step` argument only affects the outer.

**`L_out` is the standard conv formula.** With dilation 1, `L_out = (L - K) // step + 1`. PyTorch's `nn.Conv1d` uses exactly this when you pass `stride=step`.

**Why this is load-bearing for ARENA.** ARENA's `conv1d_minimal` drill assumes `stride=1`; the follow-up extension to `conv1d_general` requires this `step > 1` capability. Most learners get stuck at 'how do I represent a strided window'; the answer is exactly this two-stride trick.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()